# 05 — Audit Pass A: Availability Census

**Run only after notebook 04 (Pass B) is complete and any cohort dominance is documented.**

**Purpose:**  
- Count samples passing inclusion under Strict / Pragmatic / Broad definitions.  
- Apply go/no-go thresholds from label_spec_v0.9.md §6.  
- Identify the recommended primary definition (strictest that clears all thresholds).  
- Report sensitivity: counts excluding preprint-only labels.

**Output:** `data/processed/audit_passA_report.md`

**After this notebook:** pause for human review. Do not proceed to DE or modeling.

In [1]:
import sys, random, time
from pathlib import Path

random.seed(42)
RUN_TIMESTAMP = time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())

_cwd = Path('.').resolve()
ROOT = _cwd if (_cwd / 'src').is_dir() else _cwd.parent
assert (ROOT / 'src').is_dir(), f"Cannot locate src/ — tried {_cwd} and {_cwd.parent}"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(f'ROOT: {ROOT}')

from src.utils import log_versions
import pandas as pd

vers = log_versions()
print(f'Run: {RUN_TIMESTAMP}')

ROOT: /Users/rinatrizvanov/Desktop/Projects/Project 1 - Comp Bio for Osteosarcoma/tcab1-alt-osteo pipeline
Library versions:
  pandas: 3.0.2
  numpy: 2.4.4
  scipy: 1.17.1
  pyranges: 0.1.4
  openpyxl: 3.1.5
  requests: 2.33.1
  GEOparse: 2.0.4
Run: 2026-05-30T21:05:14Z


## Load manifest

In [2]:
MANIFEST_PATH = ROOT / 'data' / 'manifests' / 'master_manifest.csv'
manifest = pd.read_csv(MANIFEST_PATH)

for col in manifest.columns:
    if col.startswith('include_'):
        manifest[col] = manifest[col].astype(str).str.lower().map(
            {'true': True, 'false': False, '1': True, '0': False}
        ).fillna(False)

print(f'Manifest rows: {len(manifest)}')
print(manifest['alt_status'].value_counts().to_string())

Manifest rows: 115
alt_status
alt_gold_positive         60
alt_unknown               25
alt_negative_high_conf    19
alt_silver                11


## Count samples for each definition set

In [3]:
from src.audit import count_aim1, count_aim2, apply_go_nogo, GO_NOGO

all_counts  = []
all_go_nogo = []

for def_set in ('strict', 'pragmatic', 'broad'):
    c1 = count_aim1(manifest, def_set)
    c2 = count_aim2(manifest, def_set)
    merged = {**c1, **{k: v for k, v in c2.items() if k != 'definition_set'}}
    go_nogo = apply_go_nogo(merged)

    all_counts.append(merged)
    all_go_nogo.append(go_nogo)

    print(f'\n=== {def_set.upper()} ===')
    for k, v in merged.items():
        if k != 'definition_set':
            print(f'  {k}: {v}')
    for k, v in go_nogo.items():
        print(f'  verdict_{k}: {v}')


=== STRICT ===
  aim1_total: 40
  aim1_disrupted: 8
  aim1_intact: 32
  aim1_uncertain: 0
  aim1_primary_contrast: 40
  aim1_tp53_total: 12
  aim1_telo_neg_total: 0
  aim2_total: 79
  aim2_alt_positive: 60
  aim2_alt_negative: 19
  verdict_aim1_primary: MARGINAL
  verdict_aim1_tp53_sensitivity: GO
  verdict_aim2: GO (penalized only)

=== PRAGMATIC ===
  aim1_total: 40
  aim1_disrupted: 8
  aim1_intact: 32
  aim1_uncertain: 0
  aim1_primary_contrast: 40
  aim1_tp53_total: 12
  aim1_telo_neg_total: 0
  aim2_total: 90
  aim2_alt_positive: 66
  aim2_alt_negative: 24
  verdict_aim1_primary: MARGINAL
  verdict_aim1_tp53_sensitivity: GO
  verdict_aim2: GO (unpenalized)

=== BROAD ===
  aim1_total: 57
  aim1_disrupted: 9
  aim1_intact: 32
  aim1_uncertain: 16
  aim1_primary_contrast: 41
  aim1_tp53_total: 12
  aim1_telo_neg_total: 0
  aim2_total: 90
  aim2_alt_positive: 66
  aim2_alt_negative: 24
  verdict_aim1_primary: MARGINAL
  verdict_aim1_tp53_sensitivity: GO
  verdict_aim2: GO (unpenali

## Aim 1 accounting check — universe vs primary contrast

`aim1_total` is the **inclusion universe** (all rows with `include_aim1_primary_*`).
The **primary contrast** for go/no-go is `disrupted + intact` only; rows that are in the
universe but not assigned to either arm are **uncertain** (label_spec §1c).

For **Broad**, uncertain rows are usually DepMap lines included via RNA-quantile proxy
(`rna_q < 0.25` or `rna_q >= 0.75`) while `tcab1_genomic_status` is still `unknown`.
Strict / Pragmatic should show `uncertain = 0` if accounting is consistent.


In [4]:
from src.audit import count_aim1, classify_aim1_arms

print('=== Aim 1: universe vs primary contrast ===')
print(f"{'set':<10} {'universe':>8} {'disrupted':>10} {'intact':>8} {'uncertain':>10} {'contrast':>10}  check")
for def_set in ('strict', 'pragmatic', 'broad'):
    c = count_aim1(manifest, def_set)
    u, d, i, unc = c['aim1_total'], c['aim1_disrupted'], c['aim1_intact'], c['aim1_uncertain']
    pc = c['aim1_primary_contrast']
    ok = 'OK' if u == d + i + unc and pc == d + i else 'MISMATCH'
    print(f"{def_set:<10} {u:8d} {d:10d} {i:8d} {unc:10d} {pc:10d}  {ok}")

# Explicit universe filter: classify_aim1_arms already returns 'excluded' for
# rows outside the inclusion universe, but pinning the broad_mask here makes
# intent obvious at the call site and is defensive against any future
# refactor of the classifier.
broad_mask = manifest['include_aim1_primary_broad'].astype(str).str.strip().str.lower().isin(('true', '1'))
broad_arms = classify_aim1_arms(manifest, 'broad')
uncertain = manifest[broad_mask & (broad_arms == 'uncertain')]
print(f"\n=== Broad uncertain rows (n={len(uncertain)}) — not in primary contrast ===")
if len(uncertain):
    cols = ['sample_id', 'cohort', 'tcab1_genomic_status', 'tcab1_overall_status',
            'tcab1_functional_status', 'tcab1_rna_quantile_within_cohort']
    print(uncertain[cols].to_string(index=False))
    print('\nInclusion drivers (Broad uses RNA quantile when genomic status unknown):')
    for _, r in uncertain.iterrows():
        rq = r.get('tcab1_rna_quantile_within_cohort', 'unknown')
        try:
            q = float(rq)
            tag = 'low RNA (q<0.25)' if q < 0.25 else ('high RNA (q>=0.75)' if q >= 0.75 else f'RNA q={q:.2f}')
        except (TypeError, ValueError):
            tag = 'RNA unknown'
        print(f"  {r['sample_id']:18s}  {tag}; functional={r['tcab1_functional_status']}")
else:
    print('  (none)')


=== Aim 1: universe vs primary contrast ===
set        universe  disrupted   intact  uncertain   contrast  check
strict           40          8       32          0         40  OK
pragmatic        40          8       32          0         40  OK
broad            57          9       32         16         41  OK

=== Broad uncertain rows (n=16) — not in primary contrast ===
 sample_id      cohort tcab1_genomic_status tcab1_overall_status     tcab1_functional_status tcab1_rna_quantile_within_cohort
ACH-000082 DepMap_26Q1              unknown              unknown functional_intact_supported                              1.0
ACH-000613 DepMap_26Q1              unknown              unknown functional_intact_supported               0.8787878787878788
ACH-001001 DepMap_26Q1              unknown              unknown functional_intact_supported               0.8484848484848485
ACH-001712 DepMap_26Q1              unknown  candidate_disrupted    functional_loss_possible              0.12121212121212

## Sensitivity: exclude preprint-only labels

In [5]:
peer_reviewed_only = manifest[manifest['source_confidence'] != 'preprint'].copy()
# Recompute include flags for this subset
from src.label_rules import compute_inclusion_flags
flag_rows = peer_reviewed_only.apply(compute_inclusion_flags, axis=1)
for col in [c for c in manifest.columns if c.startswith('include_')]:
    peer_reviewed_only[col] = [r[col] for r in flag_rows]

print('=== SENSITIVITY: peer-reviewed labels only ===')
for def_set in ('strict', 'pragmatic', 'broad'):
    c1 = count_aim1(peer_reviewed_only, def_set)
    c2 = count_aim2(peer_reviewed_only, def_set)
    merged = {**c1, **{k: v for k, v in c2.items() if k != 'definition_set'}}
    go_nogo = apply_go_nogo(merged)
    print(f'  {def_set}: disrupted={merged.get("aim1_disrupted",0)}, '
          f'intact={merged.get("aim1_intact",0)}, '
          f'aim2_pos={merged.get("aim2_alt_positive",0)}, '
          f'aim2_neg={merged.get("aim2_alt_negative",0)}, '
          f'aim1={go_nogo.get("aim1_primary","?")}')

=== SENSITIVITY: peer-reviewed labels only ===
  strict: disrupted=4, intact=32, aim2_pos=39, aim2_neg=16, aim1=NO-GO
  pragmatic: disrupted=4, intact=32, aim2_pos=45, aim2_neg=21, aim1=NO-GO
  broad: disrupted=5, intact=32, aim2_pos=45, aim2_neg=21, aim1=NO-GO


## Select recommended definition

In [6]:
# Strictest set that clears all go/no-go thresholds
recommended = None
for counts, go_nogo in zip(all_counts, all_go_nogo):
    def_set = counts['definition_set']
    verdicts = go_nogo.values()
    all_pass = all(v.startswith('GO') or v == 'MARGINAL' for v in verdicts)
    if all_pass and recommended is None:
        recommended = def_set

print(f'Recommended primary definition: {recommended or "NONE — all fail go/no-go"}')
if recommended is None:
    print('\n⚠️  All three definition sets fail go/no-go thresholds.')
    print('Do not freeze label_spec_v1.0 until a strategy is agreed.')

Recommended primary definition: strict


## Write Pass A report

In [7]:
from src.audit import write_pass_a_report

REPORT_PATH = ROOT / 'data' / 'processed' / 'audit_passA_report.md'
write_pass_a_report(all_counts, all_go_nogo, REPORT_PATH, recommended_set=recommended)
print(f'Report: {REPORT_PATH}')

Pass A report written to /Users/rinatrizvanov/Desktop/Projects/Project 1 - Comp Bio for Osteosarcoma/tcab1-alt-osteo pipeline/data/processed/audit_passA_report.md
Report: /Users/rinatrizvanov/Desktop/Projects/Project 1 - Comp Bio for Osteosarcoma/tcab1-alt-osteo pipeline/data/processed/audit_passA_report.md


## ⛔ Hand-off checkpoint

**Do not proceed to DE, classifier, or single-cell code until:**

1. Both audit reports (`audit_passB_report.md` and `audit_passA_report.md`) are reviewed.  
2. Primary definition set is confirmed.  
3. `label_spec_v1.0.md` is frozen and committed.  

The pipeline halts here intentionally.

In [8]:
print('=' * 60)
print('FEASIBILITY AUDIT COMPLETE')
print('=' * 60)
print()
print('Results summary:')
for counts, go_nogo in zip(all_counts, all_go_nogo):
    def_set = counts['definition_set']
    print(f'  {def_set}: Aim1={go_nogo.get("aim1_primary","?")}  Aim2={go_nogo.get("aim2","?")}')
print()
print(f'Recommended primary definition: {recommended or "NONE"}')
print()
print('Next steps (human required):')
print('  1. Review audit_passB_report.md and audit_passA_report.md')
print('  2. Confirm or adjust the primary definition set')
print('  3. Freeze label_spec_v1.0.md')
print('  4. Authorize start of DE / classifier / single-cell phases')
print()
print('NO DE OR MODELING CODE HAS BEEN WRITTEN.')

FEASIBILITY AUDIT COMPLETE

Results summary:
  strict: Aim1=MARGINAL  Aim2=GO (penalized only)
  pragmatic: Aim1=MARGINAL  Aim2=GO (unpenalized)
  broad: Aim1=MARGINAL  Aim2=GO (unpenalized)

Recommended primary definition: strict

Next steps (human required):
  1. Review audit_passB_report.md and audit_passA_report.md
  2. Confirm or adjust the primary definition set
  3. Freeze label_spec_v1.0.md
  4. Authorize start of DE / classifier / single-cell phases

NO DE OR MODELING CODE HAS BEEN WRITTEN.


## Notebook summary

In [9]:
count_lines = []
for c in all_counts:
    ds = c['definition_set']
    universe = c.get('aim1_total', 0)
    pc       = c.get('aim1_primary_contrast', 0)
    d        = c.get('aim1_disrupted', 0)
    i        = c.get('aim1_intact', 0)
    unc      = c.get('aim1_uncertain', 0)
    a2_pos   = c.get('aim2_alt_positive', 0)
    a2_neg   = c.get('aim2_alt_negative', 0)
    count_lines.append(
        f'- {ds}: universe={universe}, primary_contrast={pc}, '
        f'disrupted={d}, intact={i}, uncertain={unc}, '
        f'aim2_pos={a2_pos}, aim2_neg={a2_neg}'
    )

report_lines = [
    '# Notebook 05 — Pass A Report',
    f'Run: {RUN_TIMESTAMP}',
    '',
    f'Recommended definition: {recommended or "NONE"}',
    '',
    '## Counts',
    '_universe_ = `include_aim1_primary_{set}` true; '
    '_primary_contrast_ = `disrupted + intact` (go/no-go uses `min(disrupted, intact)`); '
    '_uncertain_ = in universe but neither arm (e.g. Broad RNA-proxy with `tcab1_genomic_status` unknown).',
    '',
    *count_lines,
    '',
    '## Status: PIPELINE PAUSED — awaiting human review and label_spec_v1.0 freeze',
]
nb_report = ROOT / 'data' / 'processed' / 'notebook_05_report.md'
nb_report.write_text('\n'.join(report_lines))
print('\n'.join(report_lines))

# Notebook 05 — Pass A Report
Run: 2026-05-30T21:05:14Z

Recommended definition: strict

## Counts
_universe_ = `include_aim1_primary_{set}` true; _primary_contrast_ = `disrupted + intact` (go/no-go uses `min(disrupted, intact)`); _uncertain_ = in universe but neither arm (e.g. Broad RNA-proxy with `tcab1_genomic_status` unknown).

- strict: universe=40, primary_contrast=40, disrupted=8, intact=32, uncertain=0, aim2_pos=60, aim2_neg=19
- pragmatic: universe=40, primary_contrast=40, disrupted=8, intact=32, uncertain=0, aim2_pos=66, aim2_neg=24
- broad: universe=57, primary_contrast=41, disrupted=9, intact=32, uncertain=16, aim2_pos=66, aim2_neg=24

## Status: PIPELINE PAUSED — awaiting human review and label_spec_v1.0 freeze
